In [0]:
import pandas as pd
from pyspark.sql.functions import col, count, when

In [0]:
# =============================================================================
# LEITURA DE DADOS 
# =============================================================================

# FONTE: https://www.kaggle.com/datasets/waleedfaheem/airline-route-profitability-and-cost-analysis

TABLE_NAME = "default.airline_route_profitability"

try:

    # -------------------------------------------------------------------------
    # Leitura da tabela no catálogo do Databricks
    # -------------------------------------------------------------------------

    df = spark.table(TABLE_NAME)

    print("✅ Tabela '{TABLE_NAME}' carregada com sucesso!")

    # -------------------------------------------------------------------------
    # Informações iniciais
    # -------------------------------------------------------------------------
    
    total_rows = df.count()
    print(f"Total de registros: {total_rows}")

    display(df.limit(5))

except Exception as e:
    print(f"❌ Erro ao carregar a tabela '{TABLE_NAME}'")
    print(f"Detalhes do erro: {e}")

✅ Tabela '{TABLE_NAME}' carregada com sucesso!
Total de registros: 7974


Flight_Number,Flight_Date,Origin,Destination,Route,Aircraft_Type,Aircraft_Capacity,Passengers,Load_Factor,Flight_Hours,Season,Route_Category,Demand_Level,Ticket_Revenue,Ancillary_Revenue,Total_Revenue,Fuel_Cost,Maintenance_Cost,Crew_Cost,Depreciation_Cost,Insurance_Cost,Airport_Fees,Catering_Cost,Handling_Cost,Navigation_Fees,Sales_Distribution_Cost,Passenger_Service_Cost,Overhead_Cost,Marketing_Cost,IT_Systems_Cost,Total_Cost,Profit,Profit_Margin
EK8960,2024-12-20,DXB,ORD,DXB-ORD,Boeing 777-300ER,396,308,0.7791,14.5,Peak,Long Haul,Medium,410785.49,53431.94,464217.43,131033.88,50750.0,21750.0,65250.0,11600.0,4262.3,5185.86,4460.11,10636.9,78193.6,5100.09,88545.73,25883.92,3769.37,506421.77,-42204.34,-9.09
EK3960,2024-05-13,DXB,HYD,DXB-HYD,Boeing 787-9,296,234,0.791,4.2,Normal,Medium Haul,Medium,145890.17,17695.27,163585.44,21338.76,9240.0,5040.0,13440.0,2520.0,2016.63,5851.01,5425.84,3027.25,21586.21,6022.2,17057.61,8820.56,1937.81,123323.88,40261.57,24.61
EK7529,2024-10-12,DXB,CDG,DXB-CDG,Boeing 787-9,296,251,0.8502,7.5,Shoulder,Long Haul,High,602841.03,78724.16,681565.19,41011.77,16500.0,9000.0,24000.0,4500.0,10764.79,4134.08,3761.85,3431.7,100188.97,6514.22,31347.38,23695.34,3415.99,282266.09,399299.1,58.59
EK4543,2024-06-25,DXB,DEL,DXB-DEL,Boeing 787-9,296,229,0.7748,3.5,Low,Medium Haul,High,126485.16,17543.26,144028.42,17333.86,7700.0,4200.0,11200.0,2100.0,6508.04,6508.73,4440.93,2079.79,22686.82,4791.21,15328.62,6795.49,2245.61,113919.11,30109.31,20.91
EK3114,2024-04-20,DXB,RUH,DXB-RUH,Airbus A320,180,142,0.7901,2.2,Shoulder,Short Haul,Medium,32651.15,3871.66,36522.81,5576.8,2530.0,1716.0,3740.0,616.0,4718.58,3360.52,3414.18,1662.49,5133.62,3094.65,8252.23,1492.53,1273.27,46580.86,-10058.06,-27.54


In [0]:
df.printSchema()



root
 |-- Flight_Number: string (nullable = true)
 |-- Flight_Date: date (nullable = true)
 |-- Origin: string (nullable = true)
 |-- Destination: string (nullable = true)
 |-- Route: string (nullable = true)
 |-- Aircraft_Type: string (nullable = true)
 |-- Aircraft_Capacity: long (nullable = true)
 |-- Passengers: long (nullable = true)
 |-- Load_Factor: double (nullable = true)
 |-- Flight_Hours: double (nullable = true)
 |-- Season: string (nullable = true)
 |-- Route_Category: string (nullable = true)
 |-- Demand_Level: string (nullable = true)
 |-- Ticket_Revenue: double (nullable = true)
 |-- Ancillary_Revenue: double (nullable = true)
 |-- Total_Revenue: double (nullable = true)
 |-- Fuel_Cost: double (nullable = true)
 |-- Maintenance_Cost: double (nullable = true)
 |-- Crew_Cost: double (nullable = true)
 |-- Depreciation_Cost: double (nullable = true)
 |-- Insurance_Cost: double (nullable = true)
 |-- Airport_Fees: double (nullable = true)
 |-- Catering_Cost: double (nullabl

In [0]:
display(df.describe())

summary,Flight_Number,Origin,Destination,Route,Aircraft_Type,Aircraft_Capacity,Passengers,Load_Factor,Flight_Hours,Season,Route_Category,Demand_Level,Ticket_Revenue,Ancillary_Revenue,Total_Revenue,Fuel_Cost,Maintenance_Cost,Crew_Cost,Depreciation_Cost,Insurance_Cost,Airport_Fees,Catering_Cost,Handling_Cost,Navigation_Fees,Sales_Distribution_Cost,Passenger_Service_Cost,Overhead_Cost,Marketing_Cost,IT_Systems_Cost,Total_Cost,Profit,Profit_Margin
count,7974,7974,7974,7974,7974,7974,7974,7974,7974,7974,7974,7974,7974,7703,7974,7974,7974,7974,7974,7974,7974,7709,7718,7974,7974,7974,7974,7974,7974,7974,7974,7974
mean,null,null,null,null,null,317.80060195635815,254.81916227740155,0.801569701529968,6.327313769751608,null,null,null,264383.8737446702,33073.71719719589,297436.1976950085,46296.405877852994,18973.22485578129,8905.116879859543,25258.814898419863,4504.9456985201905,5054.911403310754,5998.7479115319775,3988.998616221815,3785.5694306496193,44659.24236393269,5721.982070479073,35742.26570604474,13448.184059443167,2930.7385013794787,225266.56375595753,72169.63393403575,6.190221971407105
stddev,null,null,null,null,null,98.7685425832912,88.1419681599549,0.08775522267632989,4.555667788644732,null,null,null,237102.21408768694,30167.51822987907,266832.89206927834,43123.03690033493,17677.194299619627,7414.772748064565,22152.943693181027,3872.411136128275,2492.832311197128,2729.957116549193,1146.7412177978426,2861.8814724384956,40575.11739520192,2295.2081117425987,29368.802113935555,12633.987522956128,1154.4643634911386,174526.32768021792,138692.75088952482,39.04058788374835
min,EK1000,DXB,AMM,DXB-AMM,Airbus A320,180,104,0.5766,1.2,Low,Long Haul,High,14812.34,1616.62,16650.02,2592.62,1380.0,936.0,2040.0,336.0,2000.79,1398.4,2000.04,483.57,2143.11,1750.11,3750.19,608.62,923.26,28019.38,-250914.85,-236.27
max,EK8973,DXB,SYD,DXB-SYD,Boeing 787-9,517,491,0.95,16.5,Shoulder,Short Haul,Medium,1303644.62,192921.91,1496566.53,191047.78,72500.0,29000.0,87000.0,14500.0,11998.99,17079.7,5999.98,13166.36,222595.18,14601.28,144736.04,78018.07,7354.16,835108.7,972133.76,64.96


In [0]:
print(df.columns)

['Flight_Number', 'Flight_Date', 'Origin', 'Destination', 'Route', 'Aircraft_Type', 'Aircraft_Capacity', 'Passengers', 'Load_Factor', 'Flight_Hours', 'Season', 'Route_Category', 'Demand_Level', 'Ticket_Revenue', 'Ancillary_Revenue', 'Total_Revenue', 'Fuel_Cost', 'Maintenance_Cost', 'Crew_Cost', 'Depreciation_Cost', 'Insurance_Cost', 'Airport_Fees', 'Catering_Cost', 'Handling_Cost', 'Navigation_Fees', 'Sales_Distribution_Cost', 'Passenger_Service_Cost', 'Overhead_Cost', 'Marketing_Cost', 'IT_Systems_Cost', 'Total_Cost', 'Profit', 'Profit_Margin']


In [0]:
print(f"Linhas totais: {total_rows}")
print(f"Linhas distintas: {df.distinct().count()}")

Linhas totais: 7974
Linhas distintas: 7974


In [0]:
from pyspark.sql.functions import col, count, when

display(
    df.select([
        count(
            when(col(c).isNull(), c)
        ).alias(c)
        for c in df.columns
    ])
)

Flight_Number,Flight_Date,Origin,Destination,Route,Aircraft_Type,Aircraft_Capacity,Passengers,Load_Factor,Flight_Hours,Season,Route_Category,Demand_Level,Ticket_Revenue,Ancillary_Revenue,Total_Revenue,Fuel_Cost,Maintenance_Cost,Crew_Cost,Depreciation_Cost,Insurance_Cost,Airport_Fees,Catering_Cost,Handling_Cost,Navigation_Fees,Sales_Distribution_Cost,Passenger_Service_Cost,Overhead_Cost,Marketing_Cost,IT_Systems_Cost,Total_Cost,Profit,Profit_Margin
0,0,0,0,0,0,0,0,0,0,0,0,0,0,271,0,0,0,0,0,0,0,265,256,0,0,0,0,0,0,0,0,0


In [0]:
from pyspark.sql.functions import when

df = df.withColumn(
    "Profitability",
    when(df.Profit_Margin >= 20, "High")
    .when(df.Profit_Margin >= 10, "Medium")
    .when(df.Profit_Margin >= 0, "Low")
    .otherwise("Loss")
)

In [0]:
display(df.limit(100))

Flight_Number,Flight_Date,Origin,Destination,Route,Aircraft_Type,Aircraft_Capacity,Passengers,Load_Factor,Flight_Hours,Season,Route_Category,Demand_Level,Ticket_Revenue,Ancillary_Revenue,Total_Revenue,Fuel_Cost,Maintenance_Cost,Crew_Cost,Depreciation_Cost,Insurance_Cost,Airport_Fees,Catering_Cost,Handling_Cost,Navigation_Fees,Sales_Distribution_Cost,Passenger_Service_Cost,Overhead_Cost,Marketing_Cost,IT_Systems_Cost,Total_Cost,Profit,Profit_Margin,Profitability
EK8960,2024-12-20,DXB,ORD,DXB-ORD,Boeing 777-300ER,396,308,0.7791,14.5,Peak,Long Haul,Medium,410785.49,53431.94,464217.43,131033.88,50750.0,21750.0,65250.0,11600.0,4262.3,5185.86,4460.11,10636.9,78193.6,5100.09,88545.73,25883.92,3769.37,506421.77,-42204.34,-9.09,Loss
EK3960,2024-05-13,DXB,HYD,DXB-HYD,Boeing 787-9,296,234,0.791,4.2,Normal,Medium Haul,Medium,145890.17,17695.27,163585.44,21338.76,9240.0,5040.0,13440.0,2520.0,2016.63,5851.01,5425.84,3027.25,21586.21,6022.2,17057.61,8820.56,1937.81,123323.88,40261.57,24.61,High
EK7529,2024-10-12,DXB,CDG,DXB-CDG,Boeing 787-9,296,251,0.8502,7.5,Shoulder,Long Haul,High,602841.03,78724.16,681565.19,41011.77,16500.0,9000.0,24000.0,4500.0,10764.79,4134.08,3761.85,3431.7,100188.97,6514.22,31347.38,23695.34,3415.99,282266.09,399299.1,58.59,High
EK4543,2024-06-25,DXB,DEL,DXB-DEL,Boeing 787-9,296,229,0.7748,3.5,Low,Medium Haul,High,126485.16,17543.26,144028.42,17333.86,7700.0,4200.0,11200.0,2100.0,6508.04,6508.73,4440.93,2079.79,22686.82,4791.21,15328.62,6795.49,2245.61,113919.11,30109.31,20.91,High
EK3114,2024-04-20,DXB,RUH,DXB-RUH,Airbus A320,180,142,0.7901,2.2,Shoulder,Short Haul,Medium,32651.15,3871.66,36522.81,5576.8,2530.0,1716.0,3740.0,616.0,4718.58,3360.52,3414.18,1662.49,5133.62,3094.65,8252.23,1492.53,1273.27,46580.86,-10058.06,-27.54,Loss
EK1737,2024-02-06,DXB,KWI,DXB-KWI,Airbus A320,180,171,0.95,2.0,Peak,Short Haul,High,55561.35,7131.73,62693.08,5169.42,2300.0,1560.0,3400.0,560.0,2723.68,3164.3,5238.84,1290.88,10373.46,4279.57,7369.82,3383.54,2002.31,52815.81,9877.27,15.75,Medium
EK2566,2024-03-27,DXB,IST,DXB-IST,Boeing 787-9,296,281,0.95,5.0,Shoulder,Medium Haul,High,197209.86,25550.13,222759.99,29311.03,11000.0,6000.0,16000.0,3000.0,4846.68,9290.85,5971.44,2336.43,32220.74,5359.55,25593.4,8024.47,3211.83,162166.43,60593.56,27.2,High
EK3653,2024-05-15,DXB,DOH,DXB-DOH,Boeing 737-800,189,148,0.7833,1.5,Normal,Short Haul,High,34738.82,4966.19,39705.01,3632.48,1800.0,1200.0,2700.0,450.0,3514.8,4628.11,2816.06,993.41,6653.68,3184.82,5579.27,1500.8,1325.46,39978.91,-273.9,-0.69,Loss
EK8367,2024-12-25,DXB,KWI,DXB-KWI,Airbus A320,180,171,0.95,2.0,Peak,Short Haul,High,57630.63,6379.85,64010.49,4521.15,2300.0,1560.0,3400.0,560.0,2852.17,4760.32,5166.24,886.93,11226.18,2883.92,6750.96,3388.79,1531.66,51788.33,12222.16,19.09,Medium
EK8561,2024-12-20,DXB,IST,DXB-IST,Boeing 787-9,296,281,0.95,5.0,Peak,Medium Haul,High,244854.25,25076.13,269930.38,26145.38,11000.0,6000.0,16000.0,3000.0,4839.69,9500.53,3049.7,3245.47,40752.59,6312.4,24894.97,14930.65,3240.82,172912.21,97018.17,35.94,High


In [0]:
df = df.withColumn(
    "Profit_Per_Hour",
    when(df.Flight_Hours > 0,
    df.Profit / df.Flight_Hours)
)

In [0]:
df = df.withColumn(
    "Revenue_Per_Passenger",
    when(df.Passengers > 0,
    df.Total_Revenue / df.Passengers)
)

In [0]:
df = df.withColumn(
    "Cost_Per_Passenger",
    when(df.Passengers > 0,
    df.Total_Cost / df.Passengers)
)

In [0]:
df = df.withColumn(
    "Profit_Per_Passenger",
    when(df.Passengers > 0,
    df.Profit / df.Passengers)
)

In [0]:
display(df.limit(100))


Flight_Number,Flight_Date,Origin,Destination,Route,Aircraft_Type,Aircraft_Capacity,Passengers,Load_Factor,Flight_Hours,Season,Route_Category,Demand_Level,Ticket_Revenue,Ancillary_Revenue,Total_Revenue,Fuel_Cost,Maintenance_Cost,Crew_Cost,Depreciation_Cost,Insurance_Cost,Airport_Fees,Catering_Cost,Handling_Cost,Navigation_Fees,Sales_Distribution_Cost,Passenger_Service_Cost,Overhead_Cost,Marketing_Cost,IT_Systems_Cost,Total_Cost,Profit,Profit_Margin,Profitability,Profit_Per_Hour,Revenue_Per_Passenger,Cost_Per_Passenger,Profit_Per_Passenger
EK8960,2024-12-20,DXB,ORD,DXB-ORD,Boeing 777-300ER,396,308,0.7791,14.5,Peak,Long Haul,Medium,410785.49,53431.94,464217.43,131033.88,50750.0,21750.0,65250.0,11600.0,4262.3,5185.86,4460.11,10636.9,78193.6,5100.09,88545.73,25883.92,3769.37,506421.77,-42204.34,-9.09,Loss,-2910.644137931034,1507.1994480519481,1644.2265259740261,-137.02707792207792
EK3960,2024-05-13,DXB,HYD,DXB-HYD,Boeing 787-9,296,234,0.791,4.2,Normal,Medium Haul,Medium,145890.17,17695.27,163585.44,21338.76,9240.0,5040.0,13440.0,2520.0,2016.63,5851.01,5425.84,3027.25,21586.21,6022.2,17057.61,8820.56,1937.81,123323.88,40261.57,24.61,High,9586.088095238094,699.083076923077,527.0251282051282,172.05799145299144
EK7529,2024-10-12,DXB,CDG,DXB-CDG,Boeing 787-9,296,251,0.8502,7.5,Shoulder,Long Haul,High,602841.03,78724.16,681565.19,41011.77,16500.0,9000.0,24000.0,4500.0,10764.79,4134.08,3761.85,3431.7,100188.97,6514.22,31347.38,23695.34,3415.99,282266.09,399299.1,58.59,High,53239.88,2715.3991633466135,1124.56609561753,1590.8330677290835
EK4543,2024-06-25,DXB,DEL,DXB-DEL,Boeing 787-9,296,229,0.7748,3.5,Low,Medium Haul,High,126485.16,17543.26,144028.42,17333.86,7700.0,4200.0,11200.0,2100.0,6508.04,6508.73,4440.93,2079.79,22686.82,4791.21,15328.62,6795.49,2245.61,113919.11,30109.31,20.91,High,8602.66,628.9450655021834,497.4633624454149,131.48170305676857
EK3114,2024-04-20,DXB,RUH,DXB-RUH,Airbus A320,180,142,0.7901,2.2,Shoulder,Short Haul,Medium,32651.15,3871.66,36522.81,5576.8,2530.0,1716.0,3740.0,616.0,4718.58,3360.52,3414.18,1662.49,5133.62,3094.65,8252.23,1492.53,1273.27,46580.86,-10058.06,-27.54,Loss,-4571.845454545454,257.2028873239436,328.0342253521127,-70.83140845070422
EK1737,2024-02-06,DXB,KWI,DXB-KWI,Airbus A320,180,171,0.95,2.0,Peak,Short Haul,High,55561.35,7131.73,62693.08,5169.42,2300.0,1560.0,3400.0,560.0,2723.68,3164.3,5238.84,1290.88,10373.46,4279.57,7369.82,3383.54,2002.31,52815.81,9877.27,15.75,Medium,4938.635,366.62619883040935,308.8643859649123,57.76181286549708
EK2566,2024-03-27,DXB,IST,DXB-IST,Boeing 787-9,296,281,0.95,5.0,Shoulder,Medium Haul,High,197209.86,25550.13,222759.99,29311.03,11000.0,6000.0,16000.0,3000.0,4846.68,9290.85,5971.44,2336.43,32220.74,5359.55,25593.4,8024.47,3211.83,162166.43,60593.56,27.2,High,12118.712,792.740177935943,577.1047330960854,215.63544483985766
EK3653,2024-05-15,DXB,DOH,DXB-DOH,Boeing 737-800,189,148,0.7833,1.5,Normal,Short Haul,High,34738.82,4966.19,39705.01,3632.48,1800.0,1200.0,2700.0,450.0,3514.8,4628.11,2816.06,993.41,6653.68,3184.82,5579.27,1500.8,1325.46,39978.91,-273.9,-0.69,Loss,-182.6,268.2770945945946,270.1277702702703,-1.8506756756756755
EK8367,2024-12-25,DXB,KWI,DXB-KWI,Airbus A320,180,171,0.95,2.0,Peak,Short Haul,High,57630.63,6379.85,64010.49,4521.15,2300.0,1560.0,3400.0,560.0,2852.17,4760.32,5166.24,886.93,11226.18,2883.92,6750.96,3388.79,1531.66,51788.33,12222.16,19.09,Medium,6111.08,374.330350877193,302.85573099415205,71.47461988304093
EK8561,2024-12-20,DXB,IST,DXB-IST,Boeing 787-9,296,281,0.95,5.0,Peak,Medium Haul,High,244854.25,25076.13,269930.38,26145.38,11000.0,6000.0,16000.0,3000.0,4839.69,9500.53,3049.7,3245.47,40752.59,6312.4,24894.97,14930.65,3240.82,172912.21,97018.17,35.94,High,19403.634,960.606334519573,615.3459430604981,345.26039145907475


In [0]:
df.write.mode("overwrite").saveAsTable(
    "default.airline_route_profitability_enriched"
)

In [0]:
display(df.groupBy("Route").avg("Profit").orderBy(col("avg(Profit)").desc()))

Route,avg(Profit)
DXB-FRA,306127.70012307697
DXB-SIN,284358.9978125001
DXB-CDG,279190.66832817334
DXB-HKG,269674.33361445786
DXB-BKK,191339.3810377359
DXB-KUL,168895.9351388889
DXB-JFK,125724.05960365848
DXB-SYD,95187.44455657496
DXB-BOM,74062.63076923082
DXB-DEL,63410.32831325307


In [0]:
display(df.groupBy("Route").avg("Profit_Margin").orderBy(col("avg(Profit_Margin)").desc()).limit(10))

Route,avg(Profit_Margin)
DXB-FRA,46.079999999999984
DXB-SIN,42.07274999999999
DXB-CDG,41.26594427244582
DXB-BKK,39.90523584905662
DXB-HKG,38.667680722891575
DXB-KUL,34.485740740740745
DXB-BOM,31.15793846153848
DXB-KHI,30.528019323671508
DXB-DEL,27.43593373493975
DXB-LHE,18.720591133004934


In [0]:
display(df.groupBy("Aircraft_Type").avg("Profit").orderBy(col("avg(Profit)").desc()))


Aircraft_Type,avg(Profit)
Airbus A380,228382.09919293816
Boeing 777-300ER,115466.68173372756
Boeing 787-9,73478.62426842675
Airbus A350-900,36806.8602235294
Airbus A320,2138.232395397488
Boeing 737-800,1017.7116322517202


In [0]:
display(df.groupBy("Demand_Level").avg("Profit_Margin"))

Demand_Level,avg(Profit_Margin)
Medium,-9.727685406698576
High,17.68687041036721


In [0]:
display(df.groupBy("Season").agg({"Profit":"avg",
                                  "Passengers":"avg"}))

Season,avg(Profit),avg(Passengers)
Peak,110278.74926573409,274.9785214785215
Normal,63480.49481509438,247.8422641509434
Shoulder,82675.08537792887,262.4848828420257
Low,25903.385017491255,229.13293353323337


In [0]:
display(df.groupBy("Route_Category").agg(
    avg((col("Fuel_Cost") / col("Total_Cost")) * 100)
)
)


Route_Category,avg(((Fuel_Cost / Total_Cost) * 100))
Long Haul,21.074286465959517
Medium Haul,18.02448574551106
Short Haul,11.83515906218431


In [0]:
display(df.groupBy("Profitability").agg(
    avg((col("Fuel_Cost") / col("Total_Cost")) * 100)
)
)

Profitability,avg(((Fuel_Cost / Total_Cost) * 100))
Loss,18.336172285559037
High,16.985713840158006
Medium,17.336966428210115
Low,17.754020841265834


In [0]:
display(df.groupBy("Profitability").avg("Load_Factor").orderBy(col("avg(Load_Factor)").desc()))

Profitability,avg(Load_Factor)
High,0.8330225121881241
Medium,0.8080988177339905
Low,0.7935385204081628
Loss,0.7606446428571411


In [0]:
display(df.groupBy("Profitability").avg("Revenue_Per_Passenger").orderBy(col("avg(Revenue_Per_Passenger)").desc()))

Profitability,avg(Revenue_Per_Passenger)
High,1428.3540123845005
Medium,912.5222775027313
Low,831.0097108444068
Loss,569.472291866656


In [0]:
display(df.groupBy("Aircraft_Type").avg("Profit_Per_Hour").orderBy(col("avg(Profit_Per_Hour)").desc()))

Aircraft_Type,avg(Profit_Per_Hour)
Airbus A380,28639.115083489858
Boeing 777-300ER,17026.791051218763
Boeing 787-9,10345.463834063874
Airbus A350-900,6556.070306745498
Airbus A320,1618.845408498406
Boeing 737-800,1133.5237536128263


In [0]:
from pyspark.sql import Row
from pyspark.sql.functions import avg

# Lista das colunas de custo
cost_columns = [
    "Fuel_Cost",
    "Maintenance_Cost",
    "Crew_Cost",
    "Depreciation_Cost",
    "Insurance_Cost",
    "Airport_Fees",
    "Catering_Cost",
    "Handling_Cost",
    "Navigation_Fees",
    "Sales_Distribution_Cost",
    "Passenger_Service_Cost",
    "Overhead_Cost",
    "Marketing_Cost",
    "IT_Systems_Cost"
]

# Calcular o percentual médio de cada custo em relação ao custo total
results = []

for cost_col in cost_columns:
    
    avg_pct = (
        df
        .select(((df[cost_col] / df["Total_Cost"]) * 100).alias("pct"))
        .agg(avg("pct").alias("avg_pct"))
        .collect()[0]["avg_pct"]
    )

    results.append(
        Row(
            Cost_Category=cost_col,
            Avg_Percentage=round(avg_pct, 2)
        )
    )

# Criar DataFrame Spark
cost_pct_df = spark.createDataFrame(results)

# Ordenar do maior para o menor percentual
cost_pct_df = cost_pct_df.orderBy(
    col("Avg_Percentage").desc()
)

cost_pct_df.write.mode("overwrite").saveAsTable(
    "default.cost_structure_analysis"
)

display(cost_pct_df)

Cost_Category,Avg_Percentage
Sales_Distribution_Cost,18.36
Fuel_Cost,17.56
Overhead_Cost,15.7
Depreciation_Cost,10.13
Maintenance_Cost,7.31
Marketing_Cost,5.52
Catering_Cost,4.28
Passenger_Service_Cost,4.09
Airport_Fees,3.8
Crew_Cost,3.78
